In [1]:
import pandas as pd
import numpy as np

In [2]:
primekg = pd.read_csv('dataverse_files/kg.csv')

host_layer = pd.read_csv('host_layer_data.csv')
drug_data = pd.read_csv('drug_data_14_07.csv')

/tmp/ipykernel_36195/2605451977.py:1: DtypeWarning: Columns (3,8) have mixed types. Specify dtype option on import or set low_memory=False.
  primekg = pd.read_csv('dataverse_files/kg.csv')


In [3]:
antibiotics_df = pd.read_csv('antibiotics_list.csv')

In [4]:
gene_listx = host_layer[host_layer['x_type'] == 'gene/protein']['x_name'].to_list()
gene_listy = host_layer[host_layer['y_type'] == 'gene/protein']['y_name'].to_list()

In [5]:
gene_list = list(set(gene_listx + gene_listy))

In [6]:
relevant_drugs = primekg[(primekg['x_type'] == 'drug') & (primekg['y_name'].isin(gene_list))].copy()

In [7]:
relevant_drugs['drug_class'] = np.zeros(len(relevant_drugs))

In [8]:
relevant_drugs[relevant_drugs['x_name'] == 'Ciprofloxacin']

,relation,display_relation,x_index,x_id,x_type,x_name,x_source,y_index,y_id,y_type,y_name,y_source,drug_class
321263,drug_protein,carrier,14188,DB00537,drug,Ciprofloxacin,DrugBank,4315,213,gene/protein,ALB,NCBI,0.0
323522,drug_protein,enzyme,14188,DB00537,drug,Ciprofloxacin,DrugBank,3955,1544,gene/protein,CYP1A2,NCBI,0.0
328080,drug_protein,target,14188,DB00537,drug,Ciprofloxacin,DrugBank,4447,3757,gene/protein,KCNH2,NCBI,0.0
333070,drug_protein,target,14188,DB00537,drug,Ciprofloxacin,DrugBank,6976,7153,gene/protein,TOP2A,NCBI,0.0
344434,drug_protein,transporter,14188,DB00537,drug,Ciprofloxacin,DrugBank,4152,5243,gene/protein,ABCB1,NCBI,0.0


In [9]:
antibiotics_df.tail(3)

,Name,Family,Usage,Popular Brand Name
102,Tedizolid,Oxazolidinone,Skin infections,Sivextro
103,Ceftaroline fosamil,Cephalosporin,"Skin infections, community-acquired pneumonia",Teflaro
104,Solithromycin,Macrolide,Community-acquired bacterial pneumonia,Solithera


In [10]:
antibiotic_name_list = antibiotics_df['Name'].to_list()

In [11]:
def drug_class_fxn(row): 
    res = row.str.contains('|'.join(antibiotic_name_list))
    if res['x_name'] == True:
        for_compoundnames = row['x_name'].split(' ')
        index = antibiotic_name_list.index(for_compoundnames[0])
        row['drug_class'] = antibiotics_df.iloc[index,1]

    return row

In [12]:
filled_drug_class = relevant_drugs.apply(lambda x: drug_class_fxn(x), axis = 1)

In [13]:
filled_drug_class = filled_drug_class[filled_drug_class['drug_class'] != 0]

In [14]:
filled_drug_class.tail()

,relation,display_relation,x_index,x_id,x_type,x_name,x_source,y_index,y_id,y_type,y_name,y_source,drug_class
345843,drug_protein,transporter,15914,DB01017,drug,Minocycline,DrugBank,13098,10864,gene/protein,SLC22A7,NCBI,Tetracycline
345908,drug_protein,transporter,14309,DB01137,drug,Levofloxacin,DrugBank,5481,6582,gene/protein,SLC22A2,NCBI,Fluoroquinolone
345932,drug_protein,transporter,15044,DB11633,drug,Isavuconazole,DrugBank,5481,6582,gene/protein,SLC22A2,NCBI,Azole
345971,drug_protein,transporter,15443,DB00520,drug,Caspofungin,DrugBank,12713,6580,gene/protein,SLC22A1,NCBI,Echinocandin
345996,drug_protein,transporter,14309,DB01137,drug,Levofloxacin,DrugBank,12713,6580,gene/protein,SLC22A1,NCBI,Fluoroquinolone


In [15]:
key_classes = drug_data['Drug_Class'].to_list()

In [16]:
#key_classes

In [17]:
filled_drug_class_key = filled_drug_class[filled_drug_class['drug_class'].str.contains('|'.join(key_classes), case = False)].copy()

In [18]:
filled_drug_class_key['drug_id'] = np.zeros(len(filled_drug_class_key))

In [19]:
filled_drug_class_key['drug_type'] = ['drug_class']*len(filled_drug_class_key)

In [20]:
filled_drug_class_key.head()

,relation,display_relation,x_index,x_id,x_type,x_name,x_source,y_index,y_id,y_type,y_name,y_source,drug_class,drug_id,drug_type
321260,drug_protein,carrier,14185,DB00512,drug,Vancomycin,DrugBank,4315,213,gene/protein,ALB,NCBI,Glycopeptide,0.0,drug_class
321263,drug_protein,carrier,14188,DB00537,drug,Ciprofloxacin,DrugBank,4315,213,gene/protein,ALB,NCBI,Fluoroquinolone,0.0,drug_class
321270,drug_protein,carrier,14195,DB00567,drug,Cephalexin,DrugBank,4315,213,gene/protein,ALB,NCBI,Cephalosporin,0.0,drug_class
321309,drug_protein,carrier,14230,DB00759,drug,Tetracycline,DrugBank,4315,213,gene/protein,ALB,NCBI,Tetracycline,0.0,drug_class
321361,drug_protein,carrier,14279,DB01015,drug,Sulfamethoxazole,DrugBank,4315,213,gene/protein,ALB,NCBI,Sulfonamide,0.0,drug_class


In [21]:
#using drug_dict from the AMR cleaning notebook for ease of ID assignment
drug_dict = {'carbapenem':'D0', 'diaminopyrimidine':'D1', 'cephalosporin':'D2',
       'tetracycline':'D3', 'fusidane':'D4',
       'lincosamide':'D5', 'sulfonamide':'D6',
       'macrolide':'D7', 'phosphonic acid':'D8',
       'disinfecting agents and antiseptics':'D9',
       'aminoglycoside':'D10', 'glycopeptide':'D11',
       'peptide':'D12', 'fluoroquinolone':'D13',
       'isoniazid-like':'D14', 'penam':'D15', 'rifamycin':'D16',
       'phenicol':'D17', 'glycylcycline':'D18', 'polyamine':'D19',
       'cephamycin':'D20', 'streptogramin A':'D21',
       'aminocoumarin':'D22', 'pyrazine':'D23',
       'nucleoside':'D24', 'streptogramin':'D25',
       'pleuromutilin':'D26', 'elfamycin':'D27',
       'oxazolidinone':'D28', 'nitrofuran':'D29',
       'mupirocin-like':'D30', 'bicyclomycin-like':'D31',
       'nitroimidazole':'D32', 'salicylic acid':'D33',
       'monobactam':'D34', 'penem':'D35', 'sulfone':'D36',
       'thioamide':'D37', 'streptogramin B':'D38',
       'nybomycin-like':'D39', 'oxacephem':'D40'}

In [22]:
filled_drug_class_key['drug_class'] = filled_drug_class_key['drug_class'].str.lower()

In [23]:
filled_drug_class_key.head()

,relation,display_relation,x_index,x_id,x_type,x_name,x_source,y_index,y_id,y_type,y_name,y_source,drug_class,drug_id,drug_type
321260,drug_protein,carrier,14185,DB00512,drug,Vancomycin,DrugBank,4315,213,gene/protein,ALB,NCBI,glycopeptide,0.0,drug_class
321263,drug_protein,carrier,14188,DB00537,drug,Ciprofloxacin,DrugBank,4315,213,gene/protein,ALB,NCBI,fluoroquinolone,0.0,drug_class
321270,drug_protein,carrier,14195,DB00567,drug,Cephalexin,DrugBank,4315,213,gene/protein,ALB,NCBI,cephalosporin,0.0,drug_class
321309,drug_protein,carrier,14230,DB00759,drug,Tetracycline,DrugBank,4315,213,gene/protein,ALB,NCBI,tetracycline,0.0,drug_class
321361,drug_protein,carrier,14279,DB01015,drug,Sulfamethoxazole,DrugBank,4315,213,gene/protein,ALB,NCBI,sulfonamide,0.0,drug_class


In [24]:
filled_drug_class_key['drug_class'] = filled_drug_class_key['drug_class'].replace({'lipoglycopeptide':'glycopeptide', 'amphenicol':'phenicol', 'polypeptide':'peptide'})

#justification
#https://www.msdmanuals.com/home/infections/antibiotics/polypeptides?_gl=1*bqonyv*_up*MQ..*_ga*MTQ5MTM0Mjk0My4xNzg0MTExNDk1*_ga_CJ792HFYYC*czE3ODQxMTE0OTQkbzEkZzAkdDE3ODQxMTE0OTQkajYwJGwwJGgw
#https://bio.libretexts.org/Bookshelves/Microbiology/Microbiology_(Kaiser)/Unit_7%3A_Microbial_Genetics_and_Microbial_Metabolism/19%3A_Review_of_Molecular_Genetics/19.1%3A_Polypeptides_and_Proteins
#https://www.msdvetmanual.com/pharmacology/antibacterial-agents/phenicols-use-in-animals
#https://en.wikipedia.org/wiki/Amphenicol

In [25]:
#Filling the main dataframe with created IDs
def id_fxn(row):
    row['drug_id'] = drug_dict[row['drug_class']]

    return row

filled_drug_class_key = filled_drug_class_key.apply(lambda x: id_fxn(x), axis = 1)

In [26]:
#checker
filled_drug_class_key[filled_drug_class_key['drug_id'] == 0]

,relation,display_relation,x_index,x_id,x_type,x_name,x_source,y_index,y_id,y_type,y_name,y_source,drug_class,drug_id,drug_type


In [27]:
filled_drug_class_key.head()

,relation,display_relation,x_index,x_id,x_type,x_name,x_source,y_index,y_id,y_type,y_name,y_source,drug_class,drug_id,drug_type
321260,drug_protein,carrier,14185,DB00512,drug,Vancomycin,DrugBank,4315,213,gene/protein,ALB,NCBI,glycopeptide,D11,drug_class
321263,drug_protein,carrier,14188,DB00537,drug,Ciprofloxacin,DrugBank,4315,213,gene/protein,ALB,NCBI,fluoroquinolone,D13,drug_class
321270,drug_protein,carrier,14195,DB00567,drug,Cephalexin,DrugBank,4315,213,gene/protein,ALB,NCBI,cephalosporin,D2,drug_class
321309,drug_protein,carrier,14230,DB00759,drug,Tetracycline,DrugBank,4315,213,gene/protein,ALB,NCBI,tetracycline,D3,drug_class
321361,drug_protein,carrier,14279,DB01015,drug,Sulfamethoxazole,DrugBank,4315,213,gene/protein,ALB,NCBI,sulfonamide,D6,drug_class


In [28]:
filled_drug_class_key = filled_drug_class_key[['relation','display_relation','drug_id','drug_type','drug_class','y_index','y_type','y_name']]

In [29]:
final_drugclass_gene_data = filled_drug_class_key.rename(columns={'drug_id':'x_index','drug_type':'x_type','drug_class':'x_name'})

In [30]:
final_drugclass_gene_data.to_csv('drugclass_gene_data.csv', index = False)